In [ ]:
import os
import json
import re
import pandas as pd
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from dotenv import load_dotenv
from groq import Groq

# Load API key
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Plot settings
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style="whitegrid")

# Paths
DATA_PATH    = "/Users/ashishjain/Documents/assignment/transcript-intelligence/data/processed/all_calls.csv"
THEMES_PATH  = "/Users/ashishjain/Documents/assignment/transcript-intelligence/data/processed/themes.json"
CHARTS_PATH  = "/Users/ashishjain/Documents/assignment/transcript-intelligence/outputs/charts/"

print("✅ Imports done")
print("✅ Groq client ready")

In [ ]:
df = pd.read_csv(DATA_PATH)

# Fix list columns
df["topics"]       = df["topics"].apply(ast.literal_eval)
df["action_items"] = df["action_items"].apply(ast.literal_eval)
df["speakers"]     = df["speakers"].apply(ast.literal_eval)

print(f"✅ Loaded       : {len(df)} calls")
print(f"📋 Columns      : {len(df.columns)}")
print(f"📞 Call Types   : {df['call_type'].value_counts().to_dict()}")
print(f"📊 Sentiments   : {df['sentiment'].value_counts().to_dict()}")

# Flatten all topics across 100 calls
all_topics    = [t for topics in df["topics"] for t in topics]
unique_topics = sorted(set(all_topics))
topic_counts  = Counter(all_topics)

print(f"📌 Total topic mentions : {len(all_topics)}")
print(f"📌 Unique topics        : {len(unique_topics)}")
print(f"\n🔝 Top 25 most frequent topics:")
for topic, count in topic_counts.most_common(25):
    bar = "█" * count
    print(f"   {topic:<35} → {count:2d}x  {bar}")

In [ ]:
# Prepare topic list
topics_text = "\n".join([f"- {t}" for t in unique_topics])

prompt = f"""
You are a senior business analyst at a B2B SaaS company.

Below are {len(unique_topics)} unique topics extracted from 100 call transcripts.
Calls are from 3 types:
- Customer Support (customers reporting issues)
- External (account managers with customers)
- Internal (team syncs, planning, engineering)

Your task:
1. Group these topics into exactly 8-10 broad business themes
2. Each theme must be meaningful to business stakeholders
3. Theme names should be short and clear (2-4 words)
4. Every topic must belong to exactly one theme
5. Think like a product or support leader reading these

Topics:
{topics_text}

Respond ONLY in valid JSON format below, no text outside JSON:
{{
  "themes": [
    {{
      "theme_name": "Theme Name Here",
      "description": "One sentence describing what this theme covers",
      "topics": ["topic1", "topic2", "topic3"]
    }}
  ]
}}
"""

print(f"✅ Prompt built")
print(f"📝 Topics included : {len(unique_topics)}")
print(f"📏 Prompt length   : {len(prompt)} chars")

In [ ]:
print("🤖 Asking Groq (llama-3.3-70b) to categorize topics...")
print("⏳ Please wait...\n")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    max_tokens=4000,
    temperature=0.2,       # low temp = consistent output
    messages=[
        {
            "role": "system",
            "content": "You are a business analyst. Always respond with valid JSON only. No markdown, no explanation outside JSON."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

raw_response = response.choices[0].message.content
print("✅ Groq responded!")
print(f"📏 Response length : {len(raw_response)} chars")
print(f"\n--- Preview (first 600 chars) ---")
print(raw_response[:600])

In [ ]:
def parse_json_response(raw):
    # Remove markdown code blocks if present
    clean = re.sub(r"```json|```", "", raw).strip()
    try:
        return json.loads(clean)
    except json.JSONDecodeError as e:
        print(f"❌ JSON parse error: {e}")
        print(f"Raw text:\n{clean[:500]}")
        return None

themes_data = parse_json_response(raw_response)

if themes_data:
    themes = themes_data["themes"]
    print(f"✅ Successfully parsed {len(themes)} themes:\n")
    for i, theme in enumerate(themes, 1):
        print(f"{i:2}. 🏷️  {theme['theme_name']}")
        print(f"     📝 {theme['description']}")
        print(f"     🔖 {len(theme['topics'])} topics: "
              f"{', '.join(theme['topics'][:4])}...")
        print()

In [ ]:
def assign_theme(call_topics, themes):
    theme_scores = {}
    for theme in themes:
        theme_set = set(t.lower().strip() for t in theme["topics"])
        call_set  = set(t.lower().strip() for t in call_topics)
        overlap   = len(theme_set & call_set)
        theme_scores[theme["theme_name"]] = overlap

    best_theme = max(theme_scores, key=theme_scores.get)
    best_score = theme_scores[best_theme]

    return best_theme if best_score > 0 else "Uncategorized"


# Apply to all calls
df["theme"] = df["topics"].apply(lambda t: assign_theme(t, themes))

print("📊 Theme Distribution (before fallback):")
for theme, count in df["theme"].value_counts().items():
    bar = "█" * count
    print(f"   {theme:<35} → {count:2d} calls  {bar}")

uncategorized_count = (df["theme"] == "Uncategorized").sum()
print(f"\n⚠️  Uncategorized: {uncategorized_count} calls")

In [ ]:
uncategorized = df[df["theme"] == "Uncategorized"]

if len(uncategorized) > 0:
    print(f"🔍 Classifying {len(uncategorized)} uncategorized calls...\n")
    theme_names   = [t["theme_name"] for t in themes]
    themes_list   = "\n".join([f"- {t['theme_name']}: {t['description']}"
                                for t in themes])

    for idx, row in uncategorized.iterrows():
        fallback_prompt = f"""
Available themes:
{themes_list}

Call title   : {row['title']}
Call summary : {row['summary'][:400]}

Which single theme name from the list above best fits this call?
Reply with ONLY the exact theme name, nothing else.
"""
        resp = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            max_tokens=30,
            temperature=0,
            messages=[{"role": "user", "content": fallback_prompt}]
        )

        assigned = resp.choices[0].message.content.strip()

        # Validate
        if assigned in theme_names:
            df.at[idx, "theme"] = assigned
            print(f"   ✅ '{row['title'][:45]}' → {assigned}")
        else:
            # Pick closest matching theme
            df.at[idx, "theme"] = theme_names[0]
            print(f"   ⚠️  '{row['title'][:45]}' → {theme_names[0]} (default)")
else:
    print("✅ No uncategorized calls — all assigned!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Theme distribution overall
theme_counts = df["theme"].value_counts()
colors = sns.color_palette("viridis", len(theme_counts))

axes[0].barh(theme_counts.index[::-1],
             theme_counts.values[::-1], color=colors[::-1])
axes[0].set_title("Calls per Theme", fontweight="bold")
axes[0].set_xlabel("Number of Calls")
for i, v in enumerate(theme_counts.values[::-1]):
    axes[0].text(v + 0.1, i, str(v), va="center", fontweight="bold")

# Theme by call type heatmap
theme_type = pd.crosstab(df["theme"], df["call_type"])
sns.heatmap(theme_type, annot=True, fmt="d", cmap="YlOrRd",
            ax=axes[1], linewidths=0.5)
axes[1].set_title("Theme × Call Type Heatmap", fontweight="bold")
axes[1].set_xlabel("Call Type")
axes[1].set_ylabel("Theme")
plt.setp(axes[1].get_xticklabels(), rotation=15)
plt.setp(axes[1].get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(f"{CHARTS_PATH}topic_themes.png", bbox_inches="tight")
plt.show()
print("✅ Chart saved → outputs/charts/topic_themes.png")

In [ ]:
print("=" * 65)
print("📋 TRANSCRIPT EXAMPLES PER THEME")
print("=" * 65)

for theme_name in df["theme"].value_counts().index:
    theme_calls = df[df["theme"] == theme_name]
    theme_info  = next((t for t in themes
                        if t["theme_name"] == theme_name), None)

    print(f"\n{'─'*65}")
    print(f"🏷️  {theme_name.upper()}  ({len(theme_calls)} calls)")
    if theme_info:
        print(f"📝 {theme_info['description']}")
    print(f"{'─'*65}")

    for _, row in theme_calls.head(2).iterrows():
        print(f"\n  📞 {row['title']}")
        print(f"     Type      : {row['call_type']}")
        print(f"     Sentiment : {row['sentiment']} "
              f"(score: {row['sentiment_score']})")
        print(f"     Topics    : {', '.join(row['topics'][:4])}")
        print(f"     Summary   : {row['summary'][:150]}...")

In [ ]:
# Save themes JSON
with open(THEMES_PATH, "w") as f:
    json.dump(themes_data, f, indent=2)

# Save updated DataFrame with theme column
df.to_csv(DATA_PATH, index=False)

print("✅ Themes saved  → data/processed/themes.json")
print("✅ DataFrame     → data/processed/all_calls.csv")

print(f"\n{'='*55}")
print("📊 FINAL THEME SUMMARY")
print(f"{'='*55}")
print(f"Total themes identified : {len(themes)}")
print(f"Total calls categorized : {len(df)}")
print(f"Uncategorized remaining : {(df['theme'] == 'Uncategorized').sum()}")
print(f"\n📊 Final Distribution:")
for theme, count in df["theme"].value_counts().items():
    pct = count / len(df) * 100
    bar = "█" * count
    print(f"   {theme:<35} → {count:2d} calls ({pct:.0f}%)  {bar}")

print("\n✅ Topic Categorization Complete!")
print("🚀 Ready for 04_sentiment_analysis.ipynb")